In [17]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [18]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [39]:
def split_datasets_intersection(ace, disc, split_date, scenario):
    """
    Разделяет два датасета ace и discover на train/test по заданной дате split_date.
    При необходимости выравнивает временные индексы (пересечение) и удаляет строки с NaN.

    Сценарии:
    1. 'ace-ace'   — обучение и тестирование на ACE (train до split_date, test после)
    2. 'ace-disc'  — обучение на ACE (до split_date), тестирование на DISC (после split_date)
    3. 'disc-ace'  — обучение на DISC (до split_date), тестирование на ACE (после split_date)
    4. 'disc-disc' — обучение и тестирование на DISC (train до split_date, test после)
    """
    a = ace.copy()
    d = disc.copy()

    # Дата появления DISC
    discover_start = pd.Timestamp('2016-07-27')

    # Сценарии
    if scenario == 'ace-ace':
        train = a[a.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'ace-disc':
        train = a[a.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    elif scenario == 'disc-ace':
        common_idx = a.index.intersection(d.index)
        a_aligned = a.loc[common_idx].dropna(axis=0, how='any')
        d_aligned = d.loc[common_idx].dropna(axis=0, how='any')
        
        train = d_aligned[d_aligned.index < split_date].copy()
        test  = a_aligned[a_aligned.index >= split_date].copy()

    elif scenario == 'disc-disc':
        common_idx = a.index.intersection(d.index)
        d_aligned = d.loc[common_idx].dropna(axis=0, how='any')
        
        train = d_aligned[d_aligned.index < split_date].copy()
        test  = d_aligned[d_aligned.index >= split_date].copy()

    else:
        raise ValueError(f"Некорректный сценарий '{scenario}'. "
                         f"Должен быть один из: 'ace-ace', 'ace-disc', 'disc-ace', 'disc-disc'")

    return train, test

In [20]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    # models['LGBM'] = Pipeline([
    #     ("boost", LGBMRegressor(
    #         metric='rmse',
    #         # objective='huber',
    #         random_state=random_state,
    #         n_estimators=1000,
    #         reg_alpha=0.2,
    #         reg_lambda=0.2,
    #         colsample_bytree=0.8,
    #         subsample=0.8,
    #         learning_rate=0.05,
    #         max_depth=6,
    #         num_leaves=10,
    #         # early_stopping_rounds=50,
    #         verbose=-1))
    # ])

    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
            boosting_type='gbdt',
            num_leaves=93,
            max_depth=5,
            learning_rate=0.014357868416776678,
            n_estimators=1160,
            # subsample_for_bin=200000,
            objective=None,
            class_weight=None,
            min_split_gain=0.0,
            min_child_weight=0.001,
            min_child_samples=25,
            subsample=0.5606239658637814,
            subsample_freq=0,
            colsample_bytree=0.9879825765791656,
            reg_alpha=0.022687699993507067,
            reg_lambda=0.005915305250069859,
            random_state=random_state,
            n_jobs=None,
            importance_type='split',
            metric='rmse',
            # early_stopping_rounds=50,
            verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(12,),
            solver='adam',
            alpha=1e-4,
            batch_size=32,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1500,
            random_state=random_state))
    ])
    return models
    
# models = build_models()

In [42]:
def evaluate_models(ace, disc, split_date, target, future_lags,
                            scenario='ace-ace', delays='24h', random_seed=42):
    """
    Один прогон по заданному сиду для всех моделей на train/test наборах, полученных из split_datasets_intersection()
    """
    models = build_models(random_state=random_seed)
    
    # Разделяем данные по сценарию
    train, test = split_datasets_intersection(ace, disc, split_date, scenario=scenario)
        
    # Определение признаков (здесь убираю из признаков все значения dst+1, dst+2, dst+3,...)
    feature_cols = [c for c in train.columns if c not in future_lags]

    train = train.dropna(subset=feature_cols + [target])
    test  = test.dropna(subset=feature_cols + [target])

    X_train, y_train = train[feature_cols].values, train[target].values
    X_test,  y_test  = test[feature_cols].values,  test[target].values
    
    run_results = []
    
    for name, model in models.items():
        try:
            # if name == 'LGBM':
            #     # Для бустинга выделяем валидационный набор
            #     X_tr, X_val, y_tr, y_val = train_test_split(
            #         X_train, y_train, test_size=0.1, random_state=random_seed)
            #     model.fit(
            #         X_tr, y_tr,
            #         boost__eval_set=[(X_val, y_val)],
            #         boost__eval_metric='l2')
            # else:
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae  = mean_absolute_error(y_test, y_pred)
            r2   = r2_score(y_test, y_pred)

            # print(f"{name:8s} seed={random_seed}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")

            run_results.append({
                'target': target,
                'delays': delays,
                'scenario': scenario,
                'forecast_model': name,
                'seed': random_seed,
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2
            })

        except Exception as e:
            print(f"{name:8s} seed={random_seed} -> Ошибка обучения: {e}")

    return run_results

In [22]:
def evaluate_models_with_error(ace, disc, split_date, target, future_lags,
                               scenario='ace-ace', delays='24h',
                               n_seeds=5, base_seed=42, results_list=None):
    """
    Для каждой модели:
      1. запускается обучение на n_seeds
      2. выводятся все значения
      3. считается среднее и std и записывается в results_list
    """
    if results_list is None:
        results_list = []

    seeds = [base_seed + i for i in range(n_seeds)]
    all_runs = []
    
    # собираем результаты всех сидов
    for seed in seeds:
        run_results = evaluate_models(
            ace, disc, split_date, target, future_lags,
            scenario=scenario, delays=delays, random_seed=seed
        )
        all_runs.extend(run_results)

    df_all = pd.DataFrame(all_runs)

    group_cols = ['target', 'scenario', 'forecast_model', 'delays']
    agg = df_all.groupby(group_cols).agg({
        'RMSE': ['mean', 'std'],
        'MAE':  ['mean', 'std'],
        'R2':   ['mean', 'std']
    }).reset_index()

    agg.columns = ['target', 'scenario', 'forecast_model', 'delays',
                   'RMSE_mean', 'RMSE_std',
                   'MAE_mean',  'MAE_std',
                   'R2_mean',   'R2_std']

    # print("\n=== Усредненные метрики ===")
    for _, row in agg.iterrows():
        print(f"{row['forecast_model']:8s}: "
              f"RMSE = {row['RMSE_mean']:.4f} ± {row['RMSE_std']:.4f}, "
              f"MAE = {row['MAE_mean']:.4f} ± {row['MAE_std']:.4f}, "
              f"R2 = {row['R2_mean']:.4f} ± {row['R2_std']:.4f}")

        results_list.append({
            'target': row['target'],
            'delays': row['delays'],
            'scenario': row['scenario'],
            'forecast_model': row['forecast_model'],
            'RMSE': row['RMSE_mean'],
            'RMSE_std':  row['RMSE_std'],
            'MAE':  row['MAE_mean'],
            'MAE_std':   row['MAE_std'],
            'R2':   row['R2_mean'],
            'R2_std':    row['R2_std']
        })

    return df_all, agg, results_list

In [23]:
def models_shuffle(ace_df, disc_df, split_date, target, future_lags,
                        scenario='shuffle', delays='24h', random_seed=42):
    """
    Один random_seed для всех моделей в shuffle-сценарии
    Возвращает список словарей с метриками по моделям
    """
    ace = ace_df.sort_index().copy()
    disc = disc_df.sort_index().copy()

    split_date = pd.Timestamp(split_date)
    
    ace['is_ace'] = 1
    disc['is_ace'] = 0

    ace_train = ace.loc[:split_date]
    ace_test  = ace.loc[split_date:]
    
    disc_train = disc.loc[:split_date]
    disc_test  = disc.loc[split_date:]

    combined_train = pd.concat([ace_train, disc_train], ignore_index=False)
    combined_train = combined_train.sort_values('datetime')
    
    feature_cols = [c for c in ace_train.columns if c not in future_lags]
    
    train = combined_train.dropna(subset=feature_cols + [target])
    test  = disc_test.dropna(subset=feature_cols + [target])

    X_train, y_train = train[feature_cols].values, train[target].values
    X_test,  y_test  = test[feature_cols].values,  test[target].values
    
    models = build_models(random_state=random_seed)

    run_results = []

    # print(f"\n=== {scenario.upper()} | target={target} | seed={random_seed} ===")
    for name, model in models.items():
        try:
            # if name == 'LGBM':
            #     X_tr, X_val, y_tr, y_val = train_test_split(
            #         X_train, y_train, test_size=0.1, random_state=random_seed)
            #     model.fit(
            #         X_tr, y_tr,
            #         boost__eval_set=[(X_val, y_val)],
            #         boost__eval_metric='l2')
            # else:
            model.fit(X_train, y_train) # тут нужен отступ для цикла

            y_pred = model.predict(X_test)
                   
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae  = mean_absolute_error(y_test, y_pred)
            r2   = r2_score(y_test, y_pred)

            # print(f"{name:8s} seed={random_seed}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")

            run_results.append({
                'target': target,
                'delays': delays,
                'scenario': scenario,
                'forecast_model': name,
                'seed': random_seed,
                'RMSE': rmse,
                'MAE': mae,
                'R2':  r2
            })

        except Exception as e:
            print(f"{name:8s} seed={random_seed} -> Ошибка обучения: {e}")

    return run_results

In [24]:
def models_shuffle_with_error(ace_df, disc_df, split_date, target, future_lags,
                              scenario='shuffle', delays='24h',
                              n_seeds=5, base_seed=42, results_list=None):
    """
    Для shuffle-сценария:
      1. запускает models_shuffle на n_seeds сидов
      2. выводит все значения
      3. считает среднее и std и записывает в results_list
    """
    if results_list is None:
        results_list = []

    seeds = [base_seed + i for i in range(n_seeds)]
    all_runs = []

    for seed in seeds:
        run_results = models_shuffle(
            ace_df, disc_df, split_date, target, future_lags,
            scenario=scenario, delays=delays, random_seed=seed
        )
        all_runs.extend(run_results)

    df_all = pd.DataFrame(all_runs)

    group_cols = ['target', 'scenario', 'forecast_model', 'delays']
    agg = df_all.groupby(group_cols).agg({
        'RMSE': ['mean', 'std'],
        'MAE':  ['mean', 'std'],
        'R2':   ['mean', 'std']
    }).reset_index()

    agg.columns = ['target', 'scenario', 'forecast_model', 'delays',
                   'RMSE_mean', 'RMSE_std',
                   'MAE_mean',  'MAE_std',
                   'R2_mean',   'R2_std']

    for _, row in agg.iterrows():
        print(f"{row['forecast_model']:8s}: "
              f"RMSE = {row['RMSE_mean']:.4f} ± {row['RMSE_std']:.4f}, "
              f"MAE = {row['MAE_mean']:.4f} ± {row['MAE_std']:.4f}, "
              f"R2 = {row['R2_mean']:.4f} ± {row['R2_std']:.4f}")

        results_list.append({
            'target': row['target'],
            'delays': row['delays'],
            'scenario': row['scenario'],
            'forecast_model': row['forecast_model'],
            'RMSE': row['RMSE_mean'],
            'RMSE_std':  row['RMSE_std'],
            'MAE':  row['MAE_mean'],
            'MAE_std':   row['MAE_std'],
            'R2':   row['R2_mean'],
            'R2_std':    row['R2_std']
        })

    return df_all, agg, results_list


In [60]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# to_date = "2017-01-01"
# to_date = pd.Timestamp(to_date)

# # data_ace_24_copy = [data_ace_24_copy.index <= to_date]
# # data_discover_24_copy = [data_ace_24_copy.index <= to_date]

# data_ace_24_copy = data_ace_24_copy.loc[:to_date]
# data_discover_24_copy = data_discover_24_copy.loc[:to_date]

# 1. Задание переменных для обучения
split_date = "2022-01-01" # дата по которой происходит разбиение данных на тренировочный и тестовый наборы
scenarios = ['ace-ace', 'ace-disc', 'disc-ace', 'disc-disc']
future_lags = [f'Dst_plus{i}' for i in range(1, 25)] # все сдвиги во времени вперед (задаются при обработке данных)
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3', 'Dst_plus6'] #, 'Dst_plus12', 'Dst_plus24'] # таргеты, на которые делаем прогнозы

results_list = []

In [61]:
cutoff_date = pd.Timestamp('2023-12-31 23:59:59')

data_ace_24_copy = data_ace_24_copy[data_ace_24_copy.index <= cutoff_date]
data_discover_24_copy = data_discover_24_copy[data_discover_24_copy.index <= cutoff_date]
data_ace_af_copy = data_ace_af_copy[data_ace_af_copy.index <= cutoff_date]
data_discover_af_copy = data_discover_af_copy[data_discover_af_copy.index <= cutoff_date]

In [62]:
# 2. Обучение для датасетов с глубиной - 24 часа по всем переменным с подсчетом ошибки на основании 5 итераций прогнозов
print("\n==== Depth - 24h ====")
for target in targets:
    print(f"\n==== Forecast of {target.upper()} ====")
    for scen in scenarios:
        print(f"\n=== {scen.upper()} ===")
        df_all, df_sum, results_list = evaluate_models_with_error(
            ace=data_ace_24_copy,
            disc=data_discover_24_copy,
            split_date=split_date,
            target=target,
            future_lags=future_lags,
            scenario=scen,
            delays='24h',
            n_seeds=3,
            base_seed=42,
            results_list=results_list)


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
LGBM    : RMSE = 3.4210 ± 0.0113, MAE = 2.3255 ± 0.0029, R2 = 0.9678 ± 0.0002
Lasso   : RMSE = 3.4124 ± 0.0000, MAE = 2.3987 ± 0.0000, R2 = 0.9679 ± 0.0000
Linear  : RMSE = 3.3728 ± 0.0000, MAE = 2.3670 ± 0.0000, R2 = 0.9687 ± 0.0000
MLP     : RMSE = 3.2621 ± 0.0196, MAE = 2.3191 ± 0.0189, R2 = 0.9707 ± 0.0004
Ridge   : RMSE = 3.3729 ± 0.0000, MAE = 2.3670 ± 0.0000, R2 = 0.9687 ± 0.0000

=== ACE-DISC ===
LGBM    : RMSE = 3.5411 ± 0.0122, MAE = 2.4270 ± 0.0044, R2 = 0.9633 ± 0.0003
Lasso   : RMSE = 3.3600 ± 0.0000, MAE = 2.4020 ± 0.0000, R2 = 0.9670 ± 0.0000
Linear  : RMSE = 3.3799 ± 0.0000, MAE = 2.4200 ± 0.0000, R2 = 0.9666 ± 0.0000
MLP     : RMSE = 3.7119 ± 0.2027, MAE = 2.6681 ± 0.1278, R2 = 0.9596 ± 0.0045
Ridge   : RMSE = 3.3798 ± 0.0000, MAE = 2.4199 ± 0.0000, R2 = 0.9666 ± 0.0000

=== DISC-ACE ===
LGBM    : RMSE = 4.0566 ± 0.0111, MAE = 2.5511 ± 0.0031, R2 = 0.9548 ± 0.0002
Lasso   : RMSE = 3.4775 ± 0.0000,

In [63]:
# 3. Обучение для датасетов с глубиной, заданной автокорреляционной функцией
print(f"\n==== Depth - autocorrelation function ====")
for target in targets:
    print(f"\n==== Forecast of {target.upper()} ====")
    for scen in scenarios:
        print(f"\n=== {scen.upper()} ===")
        df_all, df_sum, results_list = evaluate_models_with_error(
            ace=data_ace_af_copy,
            disc=data_discover_af_copy,
            split_date=split_date,
            target=target,
            future_lags=future_lags,
            scenario=scen,
            delays='auto_func',
            n_seeds=3,
            base_seed=42,
            results_list=results_list)


==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
LGBM    : RMSE = 3.3567 ± 0.0021, MAE = 2.3244 ± 0.0010, R2 = 0.9678 ± 0.0000
Lasso   : RMSE = 3.4325 ± 0.0000, MAE = 2.4294 ± 0.0000, R2 = 0.9663 ± 0.0000
Linear  : RMSE = 3.3982 ± 0.0000, MAE = 2.4025 ± 0.0000, R2 = 0.9670 ± 0.0000
MLP     : RMSE = 3.2802 ± 0.0161, MAE = 2.3445 ± 0.0169, R2 = 0.9692 ± 0.0003
Ridge   : RMSE = 3.3982 ± 0.0000, MAE = 2.4025 ± 0.0000, R2 = 0.9670 ± 0.0000

=== ACE-DISC ===
LGBM    : RMSE = 3.4146 ± 0.0022, MAE = 2.4004 ± 0.0019, R2 = 0.9657 ± 0.0000
Lasso   : RMSE = 3.3763 ± 0.0000, MAE = 2.4127 ± 0.0000, R2 = 0.9665 ± 0.0000
Linear  : RMSE = 3.3928 ± 0.0000, MAE = 2.4274 ± 0.0000, R2 = 0.9661 ± 0.0000
MLP     : RMSE = 3.7757 ± 0.2132, MAE = 2.6808 ± 0.0998, R2 = 0.9580 ± 0.0048
Ridge   : RMSE = 3.3927 ± 0.0000, MAE = 2.4273 ± 0.0000, R2 = 0.9661 ± 0.0000

=== DISC-ACE ===
LGBM    : RMSE = 3.9971 ± 0.0084, MAE = 2.5583 ± 0.0013, R2 = 0.9545 ± 0.0002
Lasso   : RM

In [64]:
# 4. Предсказание на перемешанных данных с признаком is_ace
print(f"\n==== Depth - 24h ====")
for target in targets:
    print(f"\n==== Forecast of {target.upper()} ====")
    res = df_all, df_sum, results_list = models_shuffle_with_error(
              ace_df=data_ace_24_copy,
              disc_df=data_discover_24_copy,
              split_date=split_date,
              target=target,
              future_lags=future_lags,
              scenario='shuffle',
              delays='24h',
              n_seeds=3,
              base_seed=42,
              results_list=results_list)


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====
LGBM    : RMSE = 3.3484 ± 0.0262, MAE = 2.3487 ± 0.0053, R2 = 0.9672 ± 0.0005
Lasso   : RMSE = 3.3280 ± 0.0000, MAE = 2.3800 ± 0.0000, R2 = 0.9676 ± 0.0000
Linear  : RMSE = 3.3041 ± 0.0000, MAE = 2.3611 ± 0.0000, R2 = 0.9681 ± 0.0000
MLP     : RMSE = 3.3278 ± 0.0246, MAE = 2.4278 ± 0.0273, R2 = 0.9676 ± 0.0005
Ridge   : RMSE = 3.3040 ± 0.0000, MAE = 2.3611 ± 0.0000, R2 = 0.9681 ± 0.0000

==== Forecast of DST_PLUS2 ====
LGBM    : RMSE = 5.0387 ± 0.0079, MAE = 3.6154 ± 0.0031, R2 = 0.9257 ± 0.0002
Lasso   : RMSE = 5.2171 ± 0.0000, MAE = 3.7594 ± 0.0000, R2 = 0.9203 ± 0.0000
Linear  : RMSE = 5.2109 ± 0.0000, MAE = 3.7548 ± 0.0000, R2 = 0.9205 ± 0.0000
MLP     : RMSE = 4.9672 ± 0.0359, MAE = 3.6253 ± 0.0319, R2 = 0.9278 ± 0.0010
Ridge   : RMSE = 5.2109 ± 0.0000, MAE = 3.7548 ± 0.0000, R2 = 0.9205 ± 0.0000

==== Forecast of DST_PLUS3 ====
LGBM    : RMSE = 6.4807 ± 0.0015, MAE = 4.6455 ± 0.0027, R2 = 0.8770 ± 0.0001
Lasso   : RMSE = 6.7

In [65]:
print(f"\n==== Depth - autocorrelation function ====")
for target in targets:
    print(f"\n==== Forecast of {target.upper()} ====")
    res = df_all, df_sum, results_list = models_shuffle_with_error(
          ace_df=data_ace_af_copy,
          disc_df=data_discover_af_copy,
          split_date=split_date,
          target=target,
          future_lags=future_lags,
          scenario='shuffle',
          delays='auto_func',
          n_seeds=3,
          base_seed=42,
          results_list=results_list)


==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====
LGBM    : RMSE = 3.3495 ± 0.0159, MAE = 2.3482 ± 0.0054, R2 = 0.9670 ± 0.0003
Lasso   : RMSE = 3.3431 ± 0.0000, MAE = 2.3917 ± 0.0000, R2 = 0.9671 ± 0.0000
Linear  : RMSE = 3.3169 ± 0.0000, MAE = 2.3708 ± 0.0000, R2 = 0.9676 ± 0.0000
MLP     : RMSE = 3.3312 ± 0.0555, MAE = 2.4196 ± 0.0553, R2 = 0.9673 ± 0.0011
Ridge   : RMSE = 3.3168 ± 0.0000, MAE = 2.3708 ± 0.0000, R2 = 0.9676 ± 0.0000

==== Forecast of DST_PLUS2 ====
LGBM    : RMSE = 5.0110 ± 0.0052, MAE = 3.6291 ± 0.0027, R2 = 0.9261 ± 0.0002
Lasso   : RMSE = 5.2651 ± 0.0000, MAE = 3.8043 ± 0.0000, R2 = 0.9184 ± 0.0000
Linear  : RMSE = 5.2581 ± 0.0000, MAE = 3.7995 ± 0.0000, R2 = 0.9187 ± 0.0000
MLP     : RMSE = 5.0766 ± 0.0449, MAE = 3.7184 ± 0.0374, R2 = 0.9242 ± 0.0013
Ridge   : RMSE = 5.2580 ± 0.0000, MAE = 3.7995 ± 0.0000, R2 = 0.9187 ± 0.0000

==== Forecast of DST_PLUS3 ====
LGBM    : RMSE = 6.4389 ± 0.0051, MAE = 4.6614 ± 0.0043, R2 = 0.8780 ± 0.0002

In [66]:
results_df = pd.DataFrame(results_list)

In [67]:
results_df

,target,delays,scenario,forecast_model,RMSE,RMSE_std,MAE,MAE_std,R2,R2_std
0,Dst_plus1,24h,ace-ace,LGBM,3.420999,0.011292,2.325536,0.002906,0.967783,0.000213
1,Dst_plus1,24h,ace-ace,Lasso,3.412405,0.000000,2.398673,0.000000,0.967945,0.000000
2,Dst_plus1,24h,ace-ace,Linear,3.372826,0.000000,2.367002,0.000000,0.968684,0.000000
3,Dst_plus1,24h,ace-ace,MLP,3.262080,0.019618,2.319061,0.018873,0.970706,0.000352
4,Dst_plus1,24h,ace-ace,Ridge,3.372859,0.000000,2.367032,0.000000,0.968684,0.000000
...,...,...,...,...,...,...,...,...,...,...
195,Dst_plus6,auto_func,shuffle,LGBM,9.831654,0.013255,6.941631,0.007714,0.715588,0.000767
196,Dst_plus6,auto_func,shuffle,Lasso,10.063306,0.000000,7.105277,0.000000,0.702028,0.000000
197,Dst_plus6,auto_func,shuffle,Linear,10.059509,0.000000,7.103824,0.000000,0.702252,0.000000
198,Dst_plus6,auto_func,shuffle,MLP,9.947673,0.078899,6.989879,0.087171,0.708824,0.004622


In [68]:
# results_df.to_excel("../results/models_no_adaptation_split_date_2024-01-01.xlsx", index=False)

In [69]:
results_df.to_excel("../results/models_no_adaptation_test_on_2022-2024.xlsx", index=False)